## match ID 가져오기 <추가데이터 확보하기 - 2024 ~ 2025년도 매치데이터 >
* 2023년도의 매치 데이터는 라이엇 서버에서 저장하고 있지 않아서 가져올 수 없음 

In [2]:
import pandas as pd
merged_df = pd.read_csv("merged_df.csv")
merged_df["position"].value_counts()

position
mid    46
Name: count, dtype: int64

In [4]:
import requests
import pandas as pd
import time
import dotenv
import os

from datetime import datetime

dotenv.load_dotenv()

API_KEY = os.getenv("RiotAPIKey")

# =========================================================
# match ids 수집 함수
# - 2024 ~ 2026 시즌 전용
# - 솔로랭크(queue=420)만 수집
# =========================================================
def get_match_ids(
    puuid,
    start_time,
    end_time,
    total_count=1000,
    queue=420,
    max_retry=3
):

    # =====================================================
    # puuid 검증
    # =====================================================
    if pd.isna(puuid) or puuid is None or puuid == "":

        print("[INVALID PUUID]")

        return []

    headers = {
        "X-Riot-Token": API_KEY
    }

    all_match_ids = []

    # =====================================================
    # pagination
    # =====================================================
    for start in range(0, total_count, 100):

        count = min(100, total_count - start)

        url = (
            f"https://asia.api.riotgames.com"
            f"/lol/match/v5/matches/by-puuid/{puuid}/ids"
            f"?start={start}"
            f"&count={count}"
            f"&queue={queue}"
            f"&startTime={start_time}"
            f"&endTime={end_time}"
        )

        retry = 0

        while retry < max_retry:

            try:

                r = requests.get(
                    url,
                    headers=headers,
                    timeout=10
                )

                # =========================================
                # 정상 응답
                # =========================================
                if r.status_code == 200:

                    data = r.json()

                    if isinstance(data, list):

                        all_match_ids.extend(data)

                        print(
                            f"[SUCCESS] "
                            f"start={start} "
                            f"/ {len(data)}개"
                        )

                        break

                    else:

                        print("[INVALID RESPONSE]")

                        break

                # =========================================
                # 인증 실패
                # =========================================
                elif r.status_code == 401:

                    print("[401] API KEY 만료")

                    return []

                # =========================================
                # 존재하지 않음
                # =========================================
                elif r.status_code == 404:

                    print("[404] NOT FOUND")

                    break

                # =========================================
                # Rate Limit
                # =========================================
                elif r.status_code == 429:

                    retry_after = r.headers.get(
                        "Retry-After"
                    )

                    wait_time = (
                        int(retry_after)
                        if retry_after
                        else 120
                    )

                    print(
                        f"[429] RATE LIMIT "
                        f"{wait_time}초 대기"
                    )

                    time.sleep(wait_time)

                    retry += 1

                # =========================================
                # 서버 오류
                # =========================================
                elif r.status_code in [500, 502, 503, 504]:

                    print(
                        f"[SERVER ERROR] "
                        f"{r.status_code}"
                    )

                    retry += 1

                    time.sleep(10)

                # =========================================
                # 잘못된 요청
                # =========================================
                elif r.status_code == 400:

                    print("[400] BAD REQUEST")

                    break

                # =========================================
                # 기타 오류
                # =========================================
                else:

                    print(
                        f"[ERROR] STATUS: "
                        f"{r.status_code}"
                    )

                    break

            # =============================================
            # Timeout
            # =============================================
            except requests.exceptions.Timeout:

                print("[TIMEOUT] 재시도")

                retry += 1

                time.sleep(5)

            # =============================================
            # Connection Error
            # =============================================
            except requests.exceptions.ConnectionError:

                print("[CONNECTION ERROR] 재시도")

                retry += 1

                time.sleep(5)

            # =============================================
            # 기타 예외
            # =============================================
            except Exception as e:

                print(f"[EXCEPTION] {e}")

                break

        # =================================================
        # Riot API 보호
        # =================================================
        time.sleep(1.2)

    # =====================================================
    # 중복 제거 (순서 유지)
    # =====================================================
    all_match_ids = list(
        dict.fromkeys(all_match_ids)
    )

    return all_match_ids


# =========================================================
# 시즌 timestamp
# =========================================================

SEASONS = {

    2024: {
        "start": int(
            datetime(2024, 1, 1).timestamp()
        ),
        "end": int(
            datetime(2024, 12, 31, 23, 59, 59).timestamp()
        )
    },

    2025: {
        "start": int(
            datetime(2025, 1, 1).timestamp()
        ),
        "end": int(
            datetime(2025, 12, 31, 23, 59, 59).timestamp()
        )
    },

    2026: {
    "start": int(
        datetime(2026, 1, 1).timestamp()
    ),
    "end": int(
        datetime(2026, 12, 31, 23, 59, 59).timestamp()
    )
    }

}


# =========================================================
# 선수별 시즌별 match id 수집
# merged_df 안에:
# playername / puuid
# 있다고 가정
# =========================================================

all_rows = []

for idx, row in merged_df.iterrows():

    player = row["playername"]
    puuid = row["puuid"]

    print("\n=================================================")
    print(f"PLAYER: {player}")
    print("=================================================")

    for year, season_info in SEASONS.items():

        print(f"\n[{year}] 수집 시작")

        match_ids = get_match_ids(

            puuid=puuid,

            start_time=season_info["start"],

            end_time=season_info["end"],

            total_count=1000
        )

        print(
            f"[{year}] "
            f"{len(match_ids)}개 수집 완료"
        )

        for match_id in match_ids:

            all_rows.append({

                "playername": player,

                "puuid": puuid,

                "game_year": year,

                "match_id": match_id

            })


# =========================================================
# DataFrame 변환
# =========================================================

match_ids_df = pd.DataFrame(all_rows)

print("\n===================================")
print("최종 결과")
print("===================================")

print(match_ids_df.head())

print(f"\nTOTAL ROWS: {len(match_ids_df)}")

print(
    f"UNIQUE MATCHES: "
    f"{match_ids_df['match_id'].nunique()}"
)

print(
    f"UNIQUE PLAYERS: "
    f"{match_ids_df['playername'].nunique()}"
)


# =========================================================
# 저장
# =========================================================

match_ids_df.to_csv(

    "Mid_match_ids_2023_2026.csv",

    index=False,

    encoding="utf-8-sig"
)

print("\nCSV 저장 완료")


PLAYER: ShowMaker

[2024] 수집 시작
[SUCCESS] start=0 / 0개
[SUCCESS] start=100 / 0개
[SUCCESS] start=200 / 0개
[SUCCESS] start=300 / 0개
[SUCCESS] start=400 / 0개
[SUCCESS] start=500 / 0개
[SUCCESS] start=600 / 0개
[SUCCESS] start=700 / 0개
[SUCCESS] start=800 / 0개
[SUCCESS] start=900 / 0개
[2024] 0개 수집 완료

[2025] 수집 시작
[SUCCESS] start=0 / 100개
[SUCCESS] start=100 / 100개
[SUCCESS] start=200 / 100개
[SUCCESS] start=300 / 56개
[SUCCESS] start=400 / 0개
[SUCCESS] start=500 / 0개
[SUCCESS] start=600 / 0개
[SUCCESS] start=700 / 0개
[SUCCESS] start=800 / 0개
[SUCCESS] start=900 / 0개
[2025] 356개 수집 완료

[2026] 수집 시작
[SUCCESS] start=0 / 100개
[SUCCESS] start=100 / 100개
[SUCCESS] start=200 / 100개
[SUCCESS] start=300 / 100개
[SUCCESS] start=400 / 90개
[SUCCESS] start=500 / 0개
[SUCCESS] start=600 / 0개
[SUCCESS] start=700 / 0개
[SUCCESS] start=800 / 0개
[SUCCESS] start=900 / 0개
[2026] 490개 수집 완료

PLAYER: ShowMaker

[2024] 수집 시작
[SUCCESS] start=0 / 5개
[SUCCESS] start=100 / 0개
[SUCCESS] start=200 / 0개
[SUCCESS] start=300 /

In [5]:
# =========================================================
# DataFrame 변환
# =========================================================

match_df = pd.read_csv("Mid_match_ids_2023_2026.csv")

print("=" * 50)
print("최종 match 수:", len(match_df))
print("=" * 50)

최종 match 수: 15375


In [6]:
### 년도별 match 수
match_counts = match_df['game_year'].value_counts()
match_counts 

game_year
2026    8984
2025    4865
2024    1526
Name: count, dtype: int64

In [7]:
### 선수별 match 수
player_match_counts = match_df['playername'].value_counts()
player_match_counts

playername
Ucal         1534
Calix        1270
ShowMaker    1215
Fisher       1060
FATE          969
Loki          952
Clozer        928
kyeahoo       921
SeTab         905
Scout         712
VicLa         710
Quad          703
BuLLDoG       644
Roamer        622
Karis         602
Zeka          573
Faker         467
Chovy         383
Bdd           152
Pullbae        53
Name: count, dtype: int64

* 각자 데이터 수가 다른 이유는 계정이 여러개라 그런거임 

* 중복 경기 ID를 지우면 안됨 Riot API는 같은 게임에 참가자마다 다른 stat과 통계를 지니고 있기 때문임 

In [8]:
match_df.groupby('playername')['match_id'].nunique()

playername
Bdd           152
BuLLDoG       644
Calix        1270
Chovy         383
Clozer        928
FATE          969
Faker         467
Fisher       1060
Karis         602
Loki          952
Pullbae        53
Quad          703
Roamer        622
Scout         712
SeTab         905
ShowMaker    1215
Ucal         1534
VicLa         710
Zeka          573
kyeahoo       921
Name: match_id, dtype: int64

In [9]:
len(match_df['match_id'].unique()) 

14251

### 추가 데이터 분석임 
* 이제 선수 단위로 분석 시작함 
* match ID를 기준으로 API req 보내기 

### match Detail 가져오기 

In [1]:
# 파일 불러오기
import pandas as pd

match_df = pd.read_csv("Mid_match_ids_2023_2026.csv")

import os
import dotenv

dotenv.load_dotenv()

API_KEY = os.getenv("RiotAPIKey")

In [2]:
import requests
import time

def get_match_detail(
    match_id,
    max_retry=3
):

    url = (
        f"https://asia.api.riotgames.com"
        f"/lol/match/v5/matches/{match_id}"
    )

    headers = {
        "X-Riot-Token": API_KEY
    }

    retry = 0

    while retry < max_retry:

        try:

            r = requests.get(
                url,
                headers=headers,
                timeout=10
            )

            # =====================================
            # 성공
            # =====================================
            if r.status_code == 200:

                return r.json()

            # =====================================
            # Rate Limit
            # =====================================
            elif r.status_code == 429:

                retry_after = r.headers.get(
                    "Retry-After"
                )

                wait_time = (
                    int(retry_after)
                    if retry_after
                    else 120
                )

                print(
                    f"[429] {wait_time}초 대기"
                )

                time.sleep(wait_time)

                retry += 1

            # =====================================
            # 인증 오류
            # =====================================
            elif r.status_code == 401:

                print("[401] API KEY 오류")

                return None

            # =====================================
            # 존재하지 않음
            # =====================================
            elif r.status_code == 404:

                print(f"[404] {match_id}")

                return None

            # =====================================
            # 서버 오류
            # =====================================
            elif r.status_code in [500, 502, 503, 504]:

                print(
                    f"[SERVER ERROR] "
                    f"{r.status_code}"
                )

                retry += 1

                time.sleep(10)

            else:

                print(
                    f"[ERROR] {r.status_code}"
                )

                return None

        except Exception as e:

            print(f"[EXCEPTION] {e}")

            retry += 1

            time.sleep(5)

    print(f"[FAILED] {match_id}")

    return None

<!-- match_data['info']['participants'] -->

## match data API로 가져오기 
* "그 10명 중에서 내 프로 선수 puuid와 일치하는 participant만 추출"

## match data 파싱 
match_data['info']['participants']
-> 대충 이런식으로 10명의 정보가 다 들어있는데 
-> 여기서 선수들의 퍼포먼스만 볼 예정임 
-> 프로선수 participant만 추출 예정 

* 수집할 컬럼 이름 

| Oracle            | Riot match-v5                   |
| ----------------- | ------------------------------- |
| kills             | kills                           |
| deaths            | deaths                          |
| assists           | assists                         |
| visionscore       | visionScore                     |
| earnedgold        | goldEarned                      |
| cspm              | totalMinionsKilled / gameLength |
| damagetochampions | totalDamageDealtToChampions     |
| damagetaken       | totalDamageTaken                |
| wardsplaced       | wardsPlaced                     |
| wardskilled       | wardsKilled                     |


* 경기 타임라인 가져오는 함수 

In [3]:
import requests


def get_match_timeline(
    match_id,
    api_key,
    max_retry=3
):

    url = (
        f"https://asia.api.riotgames.com"
        f"/lol/match/v5/matches/{match_id}/timeline"
    )

    headers = {
        "X-Riot-Token": api_key
    }

    retry = 0

    while retry < max_retry:

        try:

            r = requests.get(
                url,
                headers=headers
            )

            if r.status_code == 200:
                return r.json()

            elif r.status_code == 429:
                time.sleep(1)

            else:
                print(
                    f"[TIMELINE ERROR] "
                    f"{match_id} / {r.status_code}"
                )
                return None

        except Exception as e:

            print(
                f"[TIMELINE EXCEPTION] "
                f"{match_id} / {e}"
            )

            retry += 1
            time.sleep(1)

    return None

In [4]:
from datetime import datetime


def extract_player_data(match_data,timeline_data,puuid):

    participants = match_data['info']['participants']

    valid_positions = {
        'TOP',
        'JUNGLE',
        'MIDDLE',
        'BOTTOM',
        'UTILITY'
    }

    # ==================================================
    # 게임 메타 정보
    # ==================================================

    info = match_data['info']

    # 패치 버전
    # 예: 15.10.682.1234 -> 15.10
    full_version = info.get('gameVersion', '')
    patch = '.'.join(full_version.split('.')[:2])

    # 게임 날짜
    # Riot는 ms timestamp 사용
    game_creation = info.get('gameCreation')

    dt = datetime.fromtimestamp(
        game_creation / 1000
    )

    game_date = dt.strftime('%Y-%m-%d')
    game_year = dt.year
    game_month = dt.month

    for p in participants:

        if p['puuid'] == puuid:

            team_pos = p.get('teamPosition')
            indiv_pos = p.get('individualPosition')

            # 정상 포지션만
            if team_pos not in valid_positions:
                return None

            # 포지션 불일치 제거
            if team_pos != indiv_pos:
                return None

            game_duration = (
                info['gameDuration']
            )

            # 리메이크 제거
            if game_duration < 600:
                return None

            # 총 CS
            cs = (
                p['totalMinionsKilled']
                + p['neutralMinionsKilled']
            )

            # ==================================================
            # 반환 데이터
            # ==================================================

            filtered_p = {

                # -------------------
                # 메타 정보
                # -------------------

                'match_id':
                    match_data['metadata']['matchId'],

                'patch':
                    patch,

                'game_date':
                    game_date,

                'game_year':
                    game_year,

                'game_month':
                    game_month,

                # -------------------
                # 기본 정보
                # -------------------

                'puuid':
                    p['puuid'],

                'championName':
                    p['championName'],

                'teamPosition':
                    p['teamPosition'],

                'win':
                    p['win'],

                'gameDuration':
                    game_duration,

                # -------------------
                # 전투
                # -------------------

                'kills':
                    p['kills'],

                'deaths':
                    p['deaths'],

                'assists':
                    p['assists'],

                'doubleKills':
                    p['doubleKills'],

                'tripleKills':
                    p['tripleKills'],

                'quadraKills':
                    p['quadraKills'],

                'pentaKills':
                    p['pentaKills'],

                # -------------------
                # 데미지
                # -------------------

                'totalDamageDealtToChampions':
                    p['totalDamageDealtToChampions'],

                'totalDamageTaken':
                    p['totalDamageTaken'],

                'damageSelfMitigated':
                    p['damageSelfMitigated'],

                'timeCCingOthers':
                    p['timeCCingOthers'],

                # -------------------
                # 시야
                # -------------------

                'visionScore':
                    p['visionScore'],

                'wardsPlaced':
                    p['wardsPlaced'],

                'wardsKilled':
                    p['wardsKilled'],

                'visionWardsBoughtInGame':
                    p['visionWardsBoughtInGame'],

                # -------------------
                # 성장
                # -------------------

                'goldEarned':
                    p['goldEarned'],

                'champLevel':
                    p['champLevel'],

                'cs':
                    cs,

                # -------------------
                # 오브젝트
                # -------------------

                'damageDealtToObjectives':
                    p['damageDealtToObjectives'],

                'damageDealtToTurrets':
                    p['damageDealtToTurrets'],

                'dragonKills':
                    p['dragonKills'],

                'baronKills':
                    p['baronKills'],

                'turretKills':
                    p['turretKills'],

                # -------------------
                # 퍼블
                # -------------------

                'firstBloodKill':
                    p['firstBloodKill'],

                'firstBloodAssist':
                    p['firstBloodAssist'],
                   
            
            }

            # ==================================================
            # 파생 변수
            # ==================================================

            minutes = game_duration / 60

            # ==================================================
            # 팀 기준 통계
            # ==================================================

            team_kills = sum(
                x['kills']
                for x in participants
                if x['teamId'] == p['teamId']
            )

            team_damage = sum(
                x['totalDamageDealtToChampions']
                for x in participants
                if x['teamId'] == p['teamId']
            )

            team_gold = sum(
                x['goldEarned']
                for x in participants
                if x['teamId'] == p['teamId']
            )

            team_damage_taken = sum(
                x['totalDamageTaken']
                for x in participants
                if x['teamId'] == p['teamId']
            )

            filtered_p['kda'] = (
                (p['kills'] + p['assists'])
                / (p['deaths'] + 1)
            )

            filtered_p['cspm'] = (
                cs / minutes
            )

            filtered_p['dpm'] = (
                p['totalDamageDealtToChampions']
                / minutes
            )

            filtered_p['earned_gpm'] = (
                p['goldEarned']
                / minutes
            )

            filtered_p['vspm'] = (
                p['visionScore']
                / minutes
            )

            filtered_p['wpm'] = (
                p['wardsPlaced']
                / minutes
            )

            filtered_p['wcpm'] = (
                p['wardsKilled']
                / minutes
            )

            filtered_p['damagetakenperminute'] = (
                p['totalDamageTaken']
                / minutes
            )

            filtered_p['damagemitigatedperminute'] = (
                p['damageSelfMitigated']
                / minutes
            )

            # ==================================================
            # 고급 파생 변수
            # ==================================================

            # 킬 관여율
            filtered_p['kp'] = (
                (p['kills'] + p['assists'])
                / max(team_kills, 1)
            )

            # 분당 킬관여
            filtered_p['kapm'] = (
                (p['kills'] + p['assists'])
                / minutes
            )

            # 팀 내 딜 비중
            filtered_p['damage_share'] = (
                p['totalDamageDealtToChampions']
                / max(team_damage, 1)
            )

            # 팀 내 골드 비중
            filtered_p['gold_share'] = (
                p['goldEarned']
                / max(team_gold, 1)
            )

            # 팀 내 탱킹 비중
            filtered_p['tank_share'] = (
                p['totalDamageTaken']
                / max(team_damage_taken, 1)
            )

            # 오브젝트 기여도
            filtered_p['objective_control_score'] = (
                p['dragonKills']
                + (p['baronKills'] * 2)
                + p['turretKills']
            )

            # 공격성 지표
            filtered_p['aggression_score'] = (
                (
                    p['kills']
                    + p['assists']
                )
                /
                max(
                    p['deaths'],
                    1
                )
            )

            # 시야 공격성
            filtered_p['vision_aggression'] = (
                (
                    p['wardsKilled']
                    + p['visionWardsBoughtInGame']
                )
                / minutes
            )

            # 생존력
            filtered_p['survival_score'] = (
                1 / (p['deaths'] + 1)
            )

            # 성장 효율
            filtered_p['gold_per_cs'] = (
                p['goldEarned']
                / max(cs, 1)
            )

            # 딜 효율
            filtered_p['damage_per_gold'] = (
                p['totalDamageDealtToChampions']
                / max(p['goldEarned'], 1)
            )

            # 시야 효율
            filtered_p['vision_per_gold'] = (
                p['visionScore']
                / max(p['goldEarned'], 1)
            )

            # ==================================================
            # 15분 라인전 지표
            # ==================================================

            try:

                frames = (
                    timeline_data['info']['frames']
                )

                # 15분 frame
                frame15 = frames[14]

                participant_frames = (
                    frame15['participantFrames']
                )

                # 현재 participant id 찾기
                participant_id = p['participantId']

                my_frame = participant_frames[
                    str(participant_id)
                ]

                # --------------------------
                # 내 15분 데이터
                # --------------------------

                goldat15 = (
                    my_frame['totalGold']
                )

                xpat15 = (
                    my_frame['xp']
                )

                csat15 = (
                    my_frame['minionsKilled']
                    + my_frame['jungleMinionsKilled']
                )

                # ==================================================
                # 상대 라이너 찾기
                # ==================================================

                enemy = None

                for op in participants:

                    if (
                        op['teamId'] != p['teamId']
                        and
                        op['teamPosition']
                        == p['teamPosition']
                    ):

                        enemy = op
                        break

                if enemy is not None:

                    enemy_pid = (
                        enemy['participantId']
                    )

                    enemy_frame = participant_frames[
                        str(enemy_pid)
                    ]

                    enemy_gold = (
                        enemy_frame['totalGold']
                    )

                    enemy_xp = (
                        enemy_frame['xp']
                    )

                    enemy_cs = (
                        enemy_frame['minionsKilled']
                        + enemy_frame[
                            'jungleMinionsKilled'
                        ]
                    )

                    golddiffat15 = (
                        goldat15 - enemy_gold
                    )

                    xpdiffat15 = (
                        xpat15 - enemy_xp
                    )

                    csdiffat15 = (
                        csat15 - enemy_cs
                    )

                else:

                    golddiffat15 = None
                    xpdiffat15 = None
                    csdiffat15 = None

                # ==================================================
                # 컬럼 추가
                # ==================================================

                filtered_p['goldat15'] = goldat15
                filtered_p['xpat15'] = xpat15
                filtered_p['csat15'] = csat15

                filtered_p['golddiffat15'] = (
                    golddiffat15
                )

                filtered_p['xpdiffat15'] = (
                    xpdiffat15
                )

                filtered_p['csdiffat15'] = (
                    csdiffat15
                )

                filtered_p['lane_dominance_score'] = (
                    (
                        golddiffat15 * 0.4
                    )
                    +
                    (
                        xpdiffat15 * 0.3
                    )
                    +
                    (
                        csdiffat15 * 0.3
                    )
                )

                filtered_p['lane_efficiency'] = (
                    (
                        goldat15
                        + xpat15
                    )
                    /
                    max(csat15, 1)
                )


            except Exception as e:

                print(
                    f"[15M ERROR] "
                    f"{match_data['metadata']['matchId']} "
                    f"/ {e}"
                )

            return filtered_p

    return None

In [5]:
import os
import pandas as pd
import time

# =====================================================
# 저장 경로
# =====================================================

CHECKPOINT_DIR = "API_checkpoint"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    CHECKPOINT_DIR,
    "미드_매치데이터_2023_2026.csv"
)

# =====================================================
# 기존 데이터 불러오기 (복구용)
# =====================================================

if os.path.exists(SAVE_PATH):

    existing_df = pd.read_csv(SAVE_PATH)

    all_player_data = existing_df.to_dict('records')

    collected_match_ids = set(
        existing_df['match_id'].unique()
    )

    print(
        f"기존 데이터 로드 완료 "
        f"({len(collected_match_ids)}개)"
    )

else:

    all_player_data = []

    collected_match_ids = set()

    print("새로운 수집 시작")

# =====================================================
# match detail 수집
# =====================================================

for idx, row in match_df.iterrows():

    try:

        match_id = row['match_id']
        puuid = row['puuid']
        playername = row['playername']

        # =============================================
        # 이미 수집한 경기 skip
        # =============================================
        if match_id in collected_match_ids:

            print(f"[SKIP] {match_id}")

            continue

        # =============================================
        # match detail 가져오기
        # =============================================
        match_data = get_match_detail(match_id)

        if match_data is None:

            print(f"[NO DATA] {match_id}")

            continue

        # =============================================
        # participant 추출
        # =============================================
        timeline_data = get_match_timeline(
            match_id,
            API_KEY
        )

        player_data = extract_player_data(
            match_data,
            timeline_data,
            puuid
        )

        if player_data is None:

            print(
                f"[NO PARTICIPANT] "
                f"{playername}"
            )

            continue

        # =============================================
        # 추가 정보 저장
        # =============================================
        player_data['playername'] = playername
        player_data['puuid'] = puuid

        all_player_data.append(player_data)

        collected_match_ids.add(match_id)

        print(
            f"[{idx}] "
            f"{playername} "
            f"- 저장 완료"
        )

        # =============================================
        # 체크포인트 저장
        # =============================================
        if idx % 50 == 0:

            checkpoint_df = pd.DataFrame(
                all_player_data
            )

            checkpoint_df.to_csv(
                SAVE_PATH,
                index=False,
                encoding='utf-8-sig'
            )

            print(
                f"[CHECKPOINT] "
                f"{len(checkpoint_df)}개 저장"
            )

        # =============================================
        # Rate limit 방지
        # =============================================
        # time.sleep(1.2)

    except Exception as e:

        print(
            f"[ERROR] "
            f"{idx} / {e}"
        )

        continue

# =====================================================
# 최종 저장
# =====================================================

final_df = pd.DataFrame(all_player_data)

final_df.to_csv(
    SAVE_PATH,
    index=False,
    encoding='utf-8-sig'
)

print("=" * 50)
print("최종 저장 완료")
print("총 데이터 수:", len(final_df))
print("=" * 50)

기존 데이터 로드 완료 (3100개)
[SKIP] KR_7999799648
[SKIP] KR_7999115545
[SKIP] KR_7999070574
[SKIP] KR_7998506656
[SKIP] KR_7998464916
[SKIP] KR_7998403965
[SKIP] KR_7998335978
[SKIP] KR_7997164321
[SKIP] KR_7997116573
[SKIP] KR_7997059634
[SKIP] KR_7996733437
[SKIP] KR_7996712221
[SKIP] KR_7996662220
[SKIP] KR_7996624818
[SKIP] KR_7996571590
[SKIP] KR_7996520720
[SKIP] KR_7996448759
[SKIP] KR_7996396597
[SKIP] KR_7996329203
[SKIP] KR_7995995581
[SKIP] KR_7995926796
[SKIP] KR_7995848048
[SKIP] KR_7995797278
[SKIP] KR_7995707717
[SKIP] KR_7995415137
[SKIP] KR_7995331372
[SKIP] KR_7995290213
[SKIP] KR_7995240654
[SKIP] KR_7995199545
[SKIP] KR_7995164478
[SKIP] KR_7994853432
[SKIP] KR_7994840018
[SKIP] KR_7994813183
[SKIP] KR_7994768895
[SKIP] KR_7994713569
[SKIP] KR_7994659739
[SKIP] KR_7994607355
[SKIP] KR_7994558543
[SKIP] KR_7994131887
[SKIP] KR_7994026996
[SKIP] KR_7992542253
[SKIP] KR_7992503588
[SKIP] KR_7992434866
[SKIP] KR_7992333985
[SKIP] KR_7992273388
[SKIP] KR_7991914122
[SKIP] KR_799

* 중간 저장할때마다 곂치는게 존재함으로 중복된 데이터에 대한 처리를 해줘야함 
* 하지만 한 게임에 같은 선수들이 있을 수 있다보니 생기는 문제임 
* 같은 경기(match_id)에 프로 여러 명이 있었던 경우
* match_id 가 곂치는 부분은 "프로 간 같은 게임 존재"하는걸로 해석을 할 수 있기에 이 값은 처리를 안하는게 맞음

* `그래도 완전 동일 row 중복이나 , 같은 게임에 플레이어 이름이 2번 들어간 경우는 오류임으로 제거해줌 `

In [6]:
df = pd.read_csv("API_checkpoint/미드_매치데이터_2023_2026.csv")
print(df.shape)

(14153, 65)


In [7]:
print("원본 데이터 크기:", df.shape)

# 1. 완전 동일 row 중복 확인
full_duplicates = df.duplicated().sum()

print("완전 동일 row 중복 수:", full_duplicates)

# 2. 같은 선수 + 같은 경기 중복 확인
player_match_duplicates = df.duplicated(
    subset=['playername', 'match_id']
).sum()

print(
    "playername + match_id 중복 수:",
    player_match_duplicates
)

# ============================================
# 중복 제거
# ============================================

df_clean = df.drop_duplicates(
    subset=['playername', 'match_id']
)

print("중복 제거 후 데이터 크기:", df_clean.shape)

# ============================================
# 제거된 row 수 확인
# ============================================

removed_rows = len(df) - len(df_clean)

print("제거된 row 수:", removed_rows)

원본 데이터 크기: (14153, 65)
완전 동일 row 중복 수: 0
playername + match_id 중복 수: 0
중복 제거 후 데이터 크기: (14153, 65)
제거된 row 수: 0


In [8]:
a = pd.read_csv("API_checkpoint/미드_매치데이터_2023_2026.csv")
b = pd.read_csv("API_checkpoint/추가미드_매치데이터_2023_2026.csv")

riot_matchdf = pd.concat([a, b], ignore_index=True)

riot_matchdf.shape

(15517, 65)

In [9]:
### 중복 제거

riot_matchdf.drop_duplicates(inplace=True)
print("중복 제거 후 크기:", riot_matchdf.shape)

### 결측값 확인 
missing_values = riot_matchdf.isnull().sum()
print("결측값 개수:\n", missing_values)

중복 제거 후 크기: (15517, 65)
결측값 개수:
 match_id                0
patch                   0
game_date               0
game_year               0
game_month              0
                       ..
xpdiffat15              8
csdiffat15              8
lane_dominance_score    8
lane_efficiency         8
playername              0
Length: 65, dtype: int64


In [10]:
# summonerName                   23751
# riotIdGameName                     0
# 요 2개가 결측값이긴한데 어짜피 riotid이름이니까 컬럼을 드랍함

# riot_matchdf.drop(columns=['summonerName', 'riotIdGameName'], inplace=True)
missing_values = riot_matchdf.isnull().sum()
print("결측값 개수:\n", missing_values)

# 저장함
riot_matchdf.to_csv("API_checkpoint/최종_미드매치데이터.csv", index=False, encoding="utf-8-sig")

결측값 개수:
 match_id                0
patch                   0
game_date               0
game_year               0
game_month              0
                       ..
xpdiffat15              8
csdiffat15              8
lane_dominance_score    8
lane_efficiency         8
playername              0
Length: 65, dtype: int64


In [13]:
## MIDDLE만 추출함 

import pandas as pd

df = pd.read_csv("./API_checkpoint/최종_미드매치데이터.csv")

mid_data = df[df["teamPosition"] == "MIDDLE"].copy()

mid_data

,match_id,patch,game_date,game_year,game_month,puuid,championName,teamPosition,win,gameDuration,...,vision_per_gold,goldat15,xpat15,csat15,golddiffat15,xpdiffat15,csdiffat15,lane_dominance_score,lane_efficiency,playername
0,KR_7999799648,15.24,2025-12-31,2025,12,rSs9HiF3cE0JCHM0tK4xFsx8QR9eyBvgYRnfP0w149sLYt...,Ryze,MIDDLE,False,1259,...,0.001820,5431.0,6803.0,125.0,-226.0,67.0,41.0,-58.0,97.872000,ShowMaker
1,KR_7999115545,15.24,2025-12-31,2025,12,rSs9HiF3cE0JCHM0tK4xFsx8QR9eyBvgYRnfP0w149sLYt...,Katarina,MIDDLE,False,1739,...,0.001267,5129.0,6937.0,118.0,192.0,-20.0,-17.0,65.7,102.254237,ShowMaker
2,KR_7999070574,15.24,2025-12-31,2025,12,rSs9HiF3cE0JCHM0tK4xFsx8QR9eyBvgYRnfP0w149sLYt...,Zoe,MIDDLE,True,934,...,0.001346,6031.0,7688.0,141.0,1391.0,955.0,31.0,852.2,97.297872,ShowMaker
3,KR_7998506656,15.24,2025-12-31,2025,12,rSs9HiF3cE0JCHM0tK4xFsx8QR9eyBvgYRnfP0w149sLYt...,Vex,MIDDLE,True,1722,...,0.001436,5641.0,7153.0,125.0,272.0,61.0,2.0,127.7,102.352000,ShowMaker
4,KR_7998464916,15.24,2025-12-31,2025,12,rSs9HiF3cE0JCHM0tK4xFsx8QR9eyBvgYRnfP0w149sLYt...,Syndra,MIDDLE,False,1481,...,0.001094,5773.0,6956.0,124.0,-52.0,-208.0,8.0,-80.8,102.653226,ShowMaker
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15512,KR_8002280867,15.24,2026-01-01,2026,1,1nAFVgp9f3FoHBdQqDjH3XCmowE2gro51eBnKKzlLdgh1N...,Orianna,MIDDLE,True,1569,...,0.001196,6036.0,7166.0,123.0,930.0,472.0,28.0,522.0,107.333333,Callme
15513,KR_8002128336,15.24,2026-01-01,2026,1,1nAFVgp9f3FoHBdQqDjH3XCmowE2gro51eBnKKzlLdgh1N...,Zeri,MIDDLE,False,915,...,0.001139,4608.0,5892.0,114.0,-3227.0,-1216.0,7.0,-1653.5,92.105263,Callme
15514,KR_8002037522,15.24,2026-01-01,2026,1,1nAFVgp9f3FoHBdQqDjH3XCmowE2gro51eBnKKzlLdgh1N...,Ryze,MIDDLE,True,915,...,0.002147,5653.0,7295.0,135.0,783.0,358.0,0.0,420.6,95.911111,Callme
15515,KR_8001228469,15.24,2026-01-01,2026,1,1nAFVgp9f3FoHBdQqDjH3XCmowE2gro51eBnKKzlLdgh1N...,Taliyah,MIDDLE,False,927,...,0.001460,5198.0,6837.0,118.0,-90.0,-160.0,-9.0,-86.7,101.991525,Callme


In [14]:
## 최종 데이터 다시 저장 
mid_data.to_csv("API_checkpoint/최종_미드매치데이터.csv", index=False, encoding="utf-8-sig")